## Multilayer Perceptrons

Implementing a simple 3 inputs, 2 hidden layers.

The code below creates a 2D array, with dimensions `n_inputs` by `n_hidden`.

In [ ]:
import numpy as np

n_records, n_inputs = features.shape

n_hidden = 2

weights_input_to_hidden = np.random.normal(0, n_inputs**-0.5, size=(n_inputs, n_hidden))

### Hidden Layer Activation

For each hidden layer unit $h_j$, we need to calculate the weighted sum of all inputs:

$$
h_j = \sum_i w_{ij} \, x_i
$$

| Symbol | Meaning |
|--------|---------|
| $h_j$ | The pre-activation value (weighted sum) for hidden unit $j$ |
| $w_{ij}$ | Weight connecting input $i$ to hidden unit $j$ |
| $x_i$ | The $i$-th input feature |
| $\sum_i$ | Sum over all input features |

### How it works

Each hidden neuron receives *all* the inputs, but weights them differently. The weighted sum $h_j$ aggregates the inputs according to what that particular neuron has learned to "look for." After computing $h_j$, you'd typically pass it through an activation function (e.g., sigmoid) to get the hidden unit's output.

> **Real-life example:** Think of each hidden neuron as a judge in a talent show. All judges see the same performance (inputs $x_i$), but each judge weights different aspects differently — one cares more about vocals ($w_{1j}$), another about stage presence ($w_{2j}$). Each judge computes their own weighted score $h_j$ before giving a final rating.

### Matrix Form

With NumPy, we can compute *all* hidden units at once using matrix multiplication:

$$
\mathbf{h} = \mathbf{x} \cdot \mathbf{W}
$$

where $\mathbf{W}$ is the `(n_inputs, n_hidden)` weight matrix created above. This single dot product replaces the explicit loop over hidden units.

## Backpropagation

### Hidden Layer Error Term

$$
\delta_j^h = \sum_k W_{jk} \, \delta_k^o \, f'(h_j)
$$

| Symbol | Meaning |
|--------|---------|
| $\delta_j^h$ | Error term for hidden unit $j$ |
| $\delta_k^o$ | Error term for output unit $k$ (already computed) |
| $W_{jk}$ | Weight connecting hidden unit $j$ to output unit $k$ |
| $f'(h_j)$ | Derivative of the activation function, evaluated at hidden unit $j$'s weighted sum |
| $\sum_k$ | Sum over all output units $k$ |

**How it works:** A hidden unit doesn't have a "true label" to compare against directly — it only affects the error *indirectly*, through the output units it feeds into. So its error term is built by collecting the error signal $\delta_k^o$ from every output unit it connects to, weighting each by the connection strength $W_{jk}$, and then scaling by how sensitive the hidden unit's own activation was ($f'(h_j)$).

> **Real-life example:** Think of a sound engineer (hidden unit) who doesn't hear the audience's complaints directly — they only hear feedback relayed through the singers (output units) they support. If singer A says "the bass was too loud" and singer B says "it was fine," the engineer's correction depends on how much each singer's mix relied on their channel ($W_{jk}$) and how touchy their own equipment was at that setting ($f'(h_j)$).

### Weight Updates

$$
\Delta w_{ij} = \eta \, \delta_j^h \, x_i
$$

$$
\Delta w_{pq} = \eta \, \delta_{\text{output}} \, V_{\text{in}}
$$

| Symbol | Meaning |
|--------|---------|
| $\Delta w_{ij}$ | Update to the weight connecting input $i$ to hidden unit $j$ |
| $\eta$ | Learning rate |
| $x_i$ | The $i$-th input feature |
| $\Delta w_{pq}$ | Generic weight update, connecting any unit $p$ to any unit $q$ |
| $\delta_{\text{output}}$ | Error term of the receiving unit $q$ |
| $V_{\text{in}}$ | Whatever value flows into that weight (an input $x_i$, or a hidden unit's output) |

**How it works:** Both lines are the *same rule* applied at different layers. $\Delta w_{ij} = \eta \delta_j^h x_i$ updates input→hidden weights using the hidden error term we just derived. The generic form $\Delta w_{pq} = \eta \delta_{\text{output}} V_{\text{in}}$ makes explicit that this pattern repeats at every layer: learning rate × (error term of the unit receiving the connection) × (value flowing in). This is exactly why backpropagation scales to deep networks — you don't need a new formula per layer, just the right $\delta$ and the right incoming value.

> **Real-life example:** Picture a relay of translators passing a message down a hallway. Each translator (weight) adjusts their phrasing based on two things: how much the *next* translator complained about their version ($\delta$ of the receiving unit) and what they originally heard ($V_{\text{in}}$). The rule for adjusting is identical whether you're the first translator in the hall or the last — only the specific error signal and incoming message differ.


### Working Through an Example

Let's walk through the steps of calculating the weight updates for a simple two-layer network. Suppose there are two input values, one hidden unit, and one output unit, with sigmoid activations on the hidden and output units. The image below depicts this network. (Note: the input values are shown as nodes at the bottom of the image, while the network's output value is shown as $\hat{y}$ at the top. The inputs themselves do not count as a layer, which is why this is considered a two-layer network.)

![Example network with weights](imgs/example.png)

Assume we're trying to fit some binary data and the target is $y = 1$. We'll start with the forward pass, first calculating the input to the hidden unit:

$$
h = \sum_i w_i x_i = 0.1 \times 0.4 - 0.2 \times 0.3 = -0.02
$$

and the output of the hidden unit:

$$
a = f(h) = \text{sigmoid}(-0.02) = 0.495
$$

Using this as the input to the output unit, the output of the network is:

$$
\hat{y} = f(W \cdot a) = \text{sigmoid}(0.1 \times 0.495) = 0.512
$$

With the network output, we can start the backwards pass to calculate the weight updates for both layers. Using the fact that for the sigmoid function $f'(W \cdot a) = f(W \cdot a)\,(1 - f(W \cdot a))$, the error term for the output unit is:

$$
\delta^o = (y - \hat{y})\, f'(W \cdot a) = (1 - 0.512) \times 0.512 \times (1 - 0.512) = 0.122
$$

Now we need to calculate the error term for the hidden unit with backpropagation. Here we'll scale the error term from the output unit by the weight $W$ connecting it to the hidden unit. For the hidden unit error term, $\delta_j^h = \sum_k W_{jk}\, \delta_k^o\, f'(h_j)$, but since we have one hidden unit and one output unit, this is much simpler:

$$
\delta^h = W \delta^o f'(h) = 0.1 \times 0.122 \times 0.495 \times (1 - 0.495) \approx 0.00305
$$

Now that we have the errors, we can calculate the gradient descent steps. The hidden-to-output weight step is the learning rate, times the output unit error, times the hidden unit activation value:

$$
\Delta W = \eta\, \delta^o a = 0.5 \times 0.122 \times 0.495 = 0.0302
$$

Then, for the input-to-hidden weights $w_i$, it's the learning rate times the hidden unit error, times the input values:

$$
\Delta w_i = \eta\, \delta^h x_i = (0.5 \times 0.00305 \times 0.1,\; 0.5 \times 0.00305 \times 0.3) = (0.0001525,\; 0.0004575)
$$

From this example, you can see one of the effects of using the sigmoid function for the activations. The maximum derivative of the sigmoid function is 0.25, so the errors in the output layer get reduced by at least 75%, and errors in the hidden layer are scaled down by at least 93.75%! If you have a lot of layers, using a sigmoid activation function will quickly reduce the weight steps to tiny values in layers near the input. This is known as the **vanishing gradient problem**. Later in the course you'll learn about other activation functions that perform better in this regard and are more commonly used in modern network architectures.


| Symbol | Meaning |
|--------|---------|
| $x_i$ | The $i$-th input value ($x_1 = 0.1$, $x_2 = 0.3$ in this example) |
| $w_i$ | Weight connecting input $i$ to the hidden unit ($w_1 = 0.4$, $w_2 = -0.2$) |
| $h$ | Weighted sum (pre-activation) feeding the hidden unit |
| $a$ | The hidden unit's activated output, $a = f(h)$ |
| $W$ | Weight connecting the hidden unit to the output unit ($W = 0.1$) |
| $y$ | The true target value |
| $\hat{y}$ | The network's predicted output |
| $f(\cdot)$ | The activation function (sigmoid, in this example) |
| $f'(\cdot)$ | Derivative of the activation function |
| $\delta^o$ | Error term of the output unit |
| $\delta^h$ | Error term of the hidden unit |
| $\eta$ | Learning rate ($\eta = 0.5$ in this example) |
| $\Delta W$ | Weight step for the hidden-to-output weight |
| $\Delta w_i$ | Weight step for the $i$-th input-to-hidden weight |